# Project 2B: Foothold EKF on RealReplay

This notebook runs the foothold-augmented EKF on a selected bag in `data/` (start with `building.bag`).


In [1]:
import numpy as np
import gtsam
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from measurement import Measurement
from real_replay import RealReplay
from foothold_filter import FootholdNavStateImuEKF


In [16]:
cm = 0.01
APPLY_HEIGHT_PRIOR = True  # Set to False for ablation runs (e.g., stair dataset).
bag_path = "data/stair.bag"
r = RealReplay(bag_path)

foot_names = r.foot_names
x0 = gtsam.NavState(gtsam.Rot3(), np.zeros(3), np.zeros(3))
rot_var_init = np.array([1e-2, 1e-2, 1e-6], dtype=float)
P0 = np.diag(
    np.concatenate([rot_var_init, [0.05] * 3, [0.05] * 3, [0.08] * (3 * len(foot_names))])
)

ekf = FootholdNavStateImuEKF(
    x0=x0,
    P0=P0,
    num_feet=len(foot_names),
    foot_names=foot_names,
    sigma_gyro=8e-4,
    sigma_acc=1e-2,
    foothold_process_sigma=1e-4,
    sigma_meas=1 * cm,
    sigma_height_prior=15 * cm,
)

landmark_est = []
p_hist = []
ypr_hist = []
forward_vel_hist = []


def ekf_callback(m: Measurement) -> None:
    if m.dt > 0.0:
        ekf.predict(m.omega_meas, m.f_meas, m.dt)
    ekf.process_contact_measurements(
        m.old_contacts,
        m.new_contacts,
        z_world=0.0,
        apply_height_prior=APPLY_HEIGHT_PRIOR,
    )

    for foot_id in sorted(m.new_contacts):
        event_label = f"{foot_id}_{m.k}"
        landmark_est.append((event_label, ekf.footholds[ekf.resolve_index(foot_id)].copy()))

    p_hist.append(np.asarray(ekf.x.position(), dtype=float))
    ypr_hist.append(np.asarray(ekf.x.attitude().ypr(), dtype=float))
    forward_vel_hist.append(
        float((ekf.x.attitude().matrix().T @ np.asarray(ekf.x.velocity(), dtype=float))[0])
    )


r.replay(ekf_callback)
r.close()

p_hist = np.asarray(p_hist)
ypr_hist = np.asarray(ypr_hist)
forward_vel_hist = np.asarray(forward_vel_hist)

print(f"Foothold-EKF samples: {len(p_hist)}")
print(f"Foothold initializations: {len(landmark_est)}")


Foothold-EKF samples: 31646
Foothold initializations: 1565


In [17]:
k = np.arange(len(p_hist))

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Position (x,y,z)", "Yaw-Pitch-Roll", "Forward velocity"))

fig.add_trace(go.Scatter(x=k, y=p_hist[:, 0], name="x"), row=1, col=1)
fig.add_trace(go.Scatter(x=k, y=p_hist[:, 1], name="y"), row=1, col=1)
fig.add_trace(go.Scatter(x=k, y=p_hist[:, 2], name="z"), row=1, col=1)

fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 0], name="yaw"), row=2, col=1)
fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 1], name="pitch"), row=2, col=1)
fig.add_trace(go.Scatter(x=k, y=ypr_hist[:, 2], name="roll"), row=2, col=1)

fig.add_trace(go.Scatter(x=k, y=forward_vel_hist, name="forward vel"), row=3, col=1)

fig.update_layout(height=800, width=1000, title="Foothold EKF on {bag_path}")
fig.update_xaxes(title_text="sample index", row=3, col=1)
fig.show()


In [15]:
if landmark_est:
    # Color order requested: red, green, blue, then a fourth distinct color.
    palette = ["#d62728", "#2ca02c", "#1f77b4", "#ff7f0e"]
    ordered_feet = list(foot_names)
    color_map = {foot: palette[i % len(palette)] for i, foot in enumerate(ordered_feet)}

    fig = go.Figure()
    for foot in ordered_feet:
        pts = np.asarray(
            [p for label, p in landmark_est if label.startswith(f"{foot}_")],
            dtype=float,
        )
        if pts.size == 0:
            continue
        fig.add_trace(
            go.Scatter(
                x=pts[:, 0],
                y=pts[:, 1],
                mode="markers",
                marker=dict(size=7, color=color_map[foot]),
                name=foot,
            )
        )

    fig.update_layout(
        height=500,
        width=900,
        title="Estimated footholds at contact starts",
        xaxis_title="x [m]",
        yaxis_title="y [m]",
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()
else:
    print("No foothold start events found.")
